We First need to Set-up our environment for this project. We have to ensure that we have all the tools and frameworks that we will be using installed

In [1]:
!pip install -q transformers datasets accelerate seqeval evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [2]:
# This is necessary for pushing our project to github
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


After downloading the required tools, we now import them.

In [3]:
import transformers as tr
import datasets as ds
import evaluate as ev
import seqeval as se
import accelerate as ac
import torch

print(tr.__version__)
print(ds.__version__)
print(ev.__version__)
#print(se.__version__)
print(ac.__version__)
print(torch.__version__)
print("GPU is active", torch.cuda.is_available())


5.0.0
4.0.0
0.4.6
1.13.0
2.11.0+cu128
GPU is active True


We need to load the dataset.

In [4]:
from datasets import load_dataset

# we need to load the skill span labelled dataset to fine tune our model.
raw_dataset = load_dataset("jjzha/skillspan")
print(raw_dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

dev.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3174 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3569 [00:00<?, ? examples/s]

{'idx': 1, 'tokens': ['Senior', 'QA', 'Engineer', '(', 'm/f/d', ')', '<ORGANIZATION>'], 'tags_skill': ['O', 'O', 'O', 'O', 'O', 'O', 'O'], 'tags_knowledge': ['O', 'O', 'O', 'O', 'O', 'O', 'O'], 'source': 'tech'}


We need to build a dictionary which stores id to label and label to id.


In [5]:
label_bucket = ["O", "B-SKILL", "I-SKILL", "B-KNOWLEDGE", "I-KNOWLEDGE"]

# we now use a dictionary to store these values
id2label = {i: label for i, label in enumerate(label_bucket)}
label2id = {v: k for k, v in id2label.items()}


We wrote a function which gives the nummeric representation of our label bucket

In [6]:
def encode_tags(data):
  # we extract the labels
  skill_tag = data["tags_skill"]
  knowlegde_tag = data["tags_knowledge"]
  merged_bucket = []

  # we now go through them (skill and knowledge) we want to index all the Bs & Is in skill as well as for knowledge
  for s, k in zip(skill_tag, knowlegde_tag):
    if s == "B":
      merged_bucket.append(label2id["B-SKILL"])
    elif s == "I":
      merged_bucket.append(label2id["I-SKILL"])
    elif k == "B":
      merged_bucket.append(label2id["B-KNOWLEDGE"])
    elif k == "I":
      merged_bucket.append(label2id["I-KNOWLEDGE"])
    else:
      merged_bucket.append(label2id["O"])

  data["ner_tags"] = merged_bucket
  return data

# we apply our encode tag to our dataset
raw_dataset = raw_dataset.map(encode_tags)
print(raw_dataset["train"][0])


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3174 [00:00<?, ? examples/s]

Map:   0%|          | 0/3569 [00:00<?, ? examples/s]

{'idx': 1, 'tokens': ['Senior', 'QA', 'Engineer', '(', 'm/f/d', ')', '<ORGANIZATION>'], 'tags_skill': ['O', 'O', 'O', 'O', 'O', 'O', 'O'], 'tags_knowledge': ['O', 'O', 'O', 'O', 'O', 'O', 'O'], 'source': 'tech', 'ner_tags': [0, 0, 0, 0, 0, 0, 0]}


We now load our tokenizer

In [7]:
from transformers import AutoTokenizer

model_checkpoint = "jjzha/jobbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/603 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

We now have to align tokens with the labels.
Purpose: When applying a tokenizer, certain words are split into sub tokens. These sub-tokens form part of the same token. Thus we want to maintain the same numeric tokens for the sub-tokens

In [8]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label = labels[word_id]
            # If label is B (id=1), convert to I (id=2) for continuation tokens
            if label % 2 == 1:
                label += 1
            new_labels.append(label)
    return new_labels


def tokenize_and_align_labels(data):
    tokenized_inputs = tokenizer(
        data["tokens"], truncation=True, is_split_into_words=True, max_length=512
    )
    all_labels = data["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs


tokenized_datasets = raw_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)
print(tokenized_datasets)

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3174 [00:00<?, ? examples/s]

Map:   0%|          | 0/3569 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4800
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3174
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3569
    })
})


We should now pad the collected data. We will use the DataCollator from the transformer library

In [9]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

We define Metric function

In [10]:
import evaluate
import numpy as np

import evaluate
import numpy as np

metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels = [
        [label_bucket[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [label_bucket[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

We now need to load the model

In [11]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: jjzha/jobbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


we need connect to collab

In [15]:
from huggingface_hub import notebook_login

notebook_login()

We configured our training parameters

In [16]:
from transformers import TrainingArguments, EarlyStoppingCallback

args = TrainingArguments(
    "bert-finetuned-skillspan",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs=10,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    push_to_hub=True,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

we now need to train our model with the trainer function from the transformer library

In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.210558,0.387688,0.500473,0.436919,0.924265
2,0.224934,0.210053,0.442762,0.594607,0.507571,0.927341
3,0.224934,0.247859,0.468762,0.571429,0.515029,0.930817
4,0.082416,0.268197,0.495068,0.617313,0.549474,0.932676
5,0.038028,0.306820,0.495676,0.596500,0.541434,0.932068
6,0.038028,0.325680,0.502722,0.611637,0.551857,0.932207
7,0.017929,0.374493,0.500392,0.603595,0.547170,0.932172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.210558,0.387688,0.500473,0.436919,0.924265
2,0.224934,0.210053,0.442762,0.594607,0.507571,0.927341
3,0.224934,0.247859,0.468762,0.571429,0.515029,0.930817
4,0.082416,0.268197,0.495068,0.617313,0.549474,0.932676
5,0.038028,0.306820,0.495676,0.596500,0.541434,0.932068
6,0.038028,0.325680,0.502722,0.611637,0.551857,0.932207
7,0.017929,0.374493,0.500392,0.603595,0.547170,0.932172
8,0.017929,0.382517,0.502587,0.597446,0.545926,0.932086


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2400, training_loss=0.07766187846660615, metrics={'train_runtime': 1334.0307, 'train_samples_per_second': 35.981, 'train_steps_per_second': 2.249, 'total_flos': 2512869243012960.0, 'train_loss': 0.07766187846660615, 'epoch': 8.0})

We now test our model

In [18]:
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)

{'eval_loss': 0.3047773838043213, 'eval_precision': 0.5186706497386109, 'eval_recall': 0.6178825622775801, 'eval_f1': 0.5639464068209501, 'eval_accuracy': 0.9381184463151676, 'eval_runtime': 12.8488, 'eval_samples_per_second': 277.769, 'eval_steps_per_second': 17.434, 'epoch': 8.0}
